# 🚀 MLOps & Deployment Interview Handbook

ML-модель не приносит пользы, пока она "лежит в ноутбуке". Этот справочник посвящен тому, как обернуть ML-модель в API (FastAPI) и упаковать все это в контейнер (Docker), чтобы передать Production-команде.

---

## Блок 1: Виртуализация vs Контейнеризация

Частый теоретический вопрос на стыке Deployment и архитектуры:

### 🖥️ Виртуализация (Виртуальные машины - VM)
- Каждая виртуалка имеет **свою собственную полноценную Операционную Систему (Guest OS)** (например, Ubuntu, Windows).
- Очень "тяжелые" (весят гигабайты), долго запускаются (минуты).
- Полная изоляция безопасности (на уровне "железа" через Hypervisor).

### 🐳 Контейнеризация (Docker)
- Контейнеры **переиспользуют ядро Операционной Системы хоста (Host OS)**.
- Внутри контейнера лежат только библиотеки (зависимости) и код приложения.
- Очень "легкие" (миллисекунды на запуск, весят мегабайты).
- Работают везде одинаково (решает проблему "А на моем компе работало!").

---
## Блок 2: Docker и Docker-Compose

- **Docker:** Платформа для запуска контейнеров.
- **Dockerfile:** Текстовый файл-рецепт. Описывает, какой базовый образ взять (например, python:3.10), какие команды выполнить (`pip install`) и как запустить приложение.
- **Docker-Compose:** Инструмент для запуска сразу нескольких связанных контейнеров (например: Контейнер с FastAPI + Контейнер с Базой Данных PostgreSQL + Контейнер с Redis).

### 📝 Пример простейшего Dockerfile для ML:

```dockerfile
# 1. Берем легкий образ Python
FROM python:3.10-slim

# 2. Указываем рабочую директорию внутри контейнера
WORKDIR /app

# 3. Копируем зависимости и устанавливаем их
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# 4. Копируем весь свой код (и веса модели)
COPY . .

# 5. Команда для запуска сервера
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
```

---
## Блок 3: FastAPI (Лучший фреймворк для ML)

FastAPI стал стандартом де-факто для MLOps благодаря своей скорости (асинхронности) и автоматической валидации типов.

### 🔎 Основные концепции:
1. **Синхронные vs Асинхронные эндпоинты:**
   - `def my_endpoint():` (Синхронный) — выполнение заблокирует поток, пока код не отработает. ML-модели (Pytorch/Sklearn) почти всегда синхронные и не поддерживают `await` (процессорные вычисления).
   - `async def my_endpoint():` (Асинхронный) — не блокирует всю систему при обращении к базе данных или другим API (I/O вычисления).
2. **Pydantic:** Библиотека для валидации данных через типизацию Python. Ты описываешь классы (Модели Запроса и Ответа), а FastAPI сам следит, чтобы юзер прислал нужные данные (текст вместо числа вызовет ошибку 422).
3. **OpenAPI (Swagger):** Мгновенная автоматическая документация интерфейса, доступная прямо в браузере (по адресу `/docs`). Ничего не нужно писать руками!
4. **Health Check:** Обязательный эндпоинт (обычно `/health`), который просто отвечает `{"status": "ok"}`. Нужен, чтобы системы вроде Kubernetes понимали, не зависло ли приложение.

In [ ]:
# === Пример скелета API для задачи ML ===
# Обратите внимание: этот код нужно запускать в отдельном .py файле через uvicorn,
# но здесь мы приводим его для понимания архитектуры.

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

# 1. Создаем приложение FastAPI
app = FastAPI(title="My ML Model API", version="1.0")

# 2. Описываем модель входящего запроса (что мы требуем от пользователя)
class PredictRequest(BaseModel):
    age: int
    income: float
    # FastAPI автоматически проверит, что age - это целое число.

# 3. Описываем модель ответа (как мы отдаем предсказания)
class PredictResponse(BaseModel):
    is_credible: bool
    probability: float

# 4. Health-check эндпоинт
@app.get("/health")
def health_check():
    return {"status": "ok", "model_loaded": True}

# 5. Полноценный эндпоинт для ML
# Мы используем def, а не async def, так как predict() модели использует CPU (блокирует цикл)
@app.post("/predict", response_model=PredictResponse)
def predict(request: PredictRequest):
    try:
        # Гипотетическая загрузка признаков в модель
        # features = [[request.age, request.income]]
        # prediction = model.predict(features)[0]
        # proba = model.predict_proba(features)[0][1]
        
        # Имитация работы
        prediction = True if request.income > 50000 else False
        proba = 0.85
        
        return PredictResponse(is_credible=prediction, probability=proba)
        
    except Exception as e:
        # 6. Обработка ошибок. 
        # Если что-то сломается внутри, мы вернем пользователю красивую 500 ошибку
        raise HTTPException(status_code=500, detail=f"Internal Model Error: {str(e)}")

### 📝 Практические вопросы для самопроверки

1. Опишите своими словами, почему ML-модели чаще запускают внутри синхронных функций (`def`), а запросы в БД внутри асинхронных (`async def`)?
2. Что будет, если в эндпоинт `/predict` на место поля `income` пользователь отправит строку `"много денег"` вместо числа?
3. Как запустить контейнер Docker из готового `Dockerfile`? (ответ: `docker build -t my-app .` -> `docker run -p 8000:8000 my-app`).

---
## 🎯 Реальные вопросы с собеседований (ML Engineer / Python Developer)

### 🎤 Теоретические вопросы по архитектуре:
1. **Долгие запросы (Long-Polling/Task Queues):** Что произойдет, если инференс вашей ML модели занимает 10 секунд, и на FastAPI придет 1000 запросов одновременно? Как это исправить архитектурно? (Ожидается ответ про Celery / RabbitMQ / BackgroundTasks и возвращение `task_id`).
2. **Dockerfile CMD vs ENTRYPOINT:** В чем разница между `CMD ["python", "app.py"]` и `ENTRYPOINT ["python", "app.py"]`?
3. **States:** Что означает термин "Stateless сервис"? Почему микросервисы (и в частности ML-инференс) принято делать stateless, и где в таком случае хранить кэш сессий (ответ: Redis).
4. **Uvicorn Workers:** Зачем в команде запуска FastAPI передают флаг `--workers 4` (или работают через Gunicorn), если Python имеет GIL?